# Q2 Data preparation

Pitch velocity within Blue Jays outings

In [1]:
import pandas as pd

raw = pd.read_csv('MLB_Data.csv')
tor_pitching = (
    ((raw['home_team'] == 'TOR') & (raw['inning_topbot'] == 'Top'))
    | ((raw['away_team'] == 'TOR') & (raw['inning_topbot'] == 'Bot'))
)
columns = [
    'game_date', 'game_pk', 'pitcher', 'player_name', 'batter', 'inning',
    'at_bat_number', 'pitch_number', 'release_speed', 'pitch_type', 'pitch_name',
    'balls', 'strikes', 'stand', 'p_throws', 'zone', 'description', 'des',
    'events', 'woba_value', 'woba_denom',
]
q2 = raw.loc[tor_pitching, columns].copy()

## Add only the measures needed

| New field | Meaning |
|---|---|
| `outing_id` | One pitcher in one game. |
| `pitch_count_outing` | Pitch position across the entire outing, not just one batter. |
| `workload_within_10` | Pitch count centered within the outing and divided by 10, for the existing LMM. |
| `count_state` | Pre-pitch balls-strikes, such as `0-2`. |
| `in_zone` | True for recorded zones 1–9; its mean gives Zone%. A location proxy, not direct command accuracy. |
| `is_swing`, `is_whiff` | Ordinary swing and swing-and-miss indicators; whiff rate is whiffs / swings. |

The short bunt condition excludes the three in-play bunts that are coded as `hit_into_play`. Called strikes and explicit foul/missed bunts are not ordinary swings.

In [2]:
q2['game_date'] = pd.to_datetime(q2['game_date'])
q2 = q2.sort_values(['game_pk', 'at_bat_number', 'pitch_number']).reset_index(drop=True)
q2['outing_id'] = q2['game_pk'].astype(str) + '_' + q2['pitcher'].astype(str)
q2['pitch_count_outing'] = q2.groupby('outing_id').cumcount() + 1
q2['workload_within_10'] = (
    q2['pitch_count_outing'] - q2.groupby('outing_id')['pitch_count_outing'].transform('mean')
) / 10
q2['count_state'] = q2['balls'].astype(str) + '-' + q2['strikes'].astype(str)
q2['in_zone'] = q2['zone'].between(1, 9)
bunt_contact = q2['description'].eq('hit_into_play') & q2['des'].str.contains('bunt', case=False, na=False)
swing_events = ['swinging_strike', 'swinging_strike_blocked', 'foul', 'foul_tip', 'hit_into_play']
q2['is_swing'] = q2['description'].isin(swing_events) & ~bunt_contact
q2['is_whiff'] = q2['description'].isin(['swinging_strike', 'swinging_strike_blocked'])

In [4]:
print('Pitches:', len(q2), '| Pitchers:', q2['pitcher'].nunique(), '| Outings:', q2['outing_id'].nunique())
q2[['player_name', 'inning', 'pitch_count_outing', 'release_speed', 'in_zone', 'is_swing', 'is_whiff']].head(20)

Pitches: 2788 | Pitchers: 15 | Outings: 101


,player_name,inning,pitch_count_outing,release_speed,in_zone,is_swing,is_whiff
0,"Yesavage, Trey",1,1,94.0,False,False,False
1,"Yesavage, Trey",1,2,84.6,True,False,False
2,"Yesavage, Trey",1,3,88.7,True,True,True
3,"Yesavage, Trey",1,4,84.8,False,False,False
4,"Yesavage, Trey",1,5,89.3,True,True,False
5,"Yesavage, Trey",1,6,95.4,False,True,False
6,"Yesavage, Trey",1,7,95.3,True,False,False
7,"Yesavage, Trey",1,8,93.7,True,True,False
8,"Yesavage, Trey",1,9,95.3,True,False,False
9,"Yesavage, Trey",1,10,89.6,False,True,True
